In [2]:
import pandas as pd
import numpy as np
import ftfy
import random
from collections import defaultdict
from datasets import Dataset, DatasetDict
from preprocess import transcriptions

/home/onyxia/work/Projet-de-NLP/preprocess.py:18: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("data/metadonnees.csv")


## Préparation des données

In [3]:
# Chargement des données et gestion des problèmes d'encodage
annotations1 = pd.read_csv("annotations/annotation_token_strategy_annotations.csv", sep=";",encoding="latin-1")
annotations1 = annotations1.apply(lambda col: col.map(lambda x: ftfy.fix_text(x) if isinstance(x, str) else x))
annotations1 = annotations1.dropna()

annotations2 = pd.read_csv("annotations/annotation_tokens_complémentaire.csv", sep=";")
annotations2 = annotations2.rename(columns = {"token":"token_text", 
                                    'Unnamed: 0':"phrase_id", 
                                    "text_id":"doc_id_origine"})
annotations2 = annotations2.dropna()

# Erreurs de frappe constatées a posteriori
annotations1["label_concret"] = annotations1["label_concret"].replace('B-VERB_CONRET', 'B-VERB_CONCRET')
annotations1["label_concret"] = annotations1["label_concret"].replace('B-VERB_CONCRET ', 'B-VERB_CONCRET')
annotations1["label_concret"] = annotations1["label_concret"].replace('I-VERB-CONCRET', 'I-VERB_CONCRET')
annotations1["label_concret"] = annotations1["label_concret"].replace('E-VERB-CONCRET', 'E-VERB_CONCRET')
annotations1["label_concret"] = annotations1["label_concret"].replace('B-VERB-CONTEXTE', 'B-VERB_CONTEXTE') 

annotations1["label_discours"] = annotations1["label_discours"].replace('B-ACT', 'B-VERB_ACT')  
annotations1["label_discours"] = annotations1["label_discours"].replace('I-VERB_ETA', 'I-VERB_ETAT') 
annotations1["label_discours"] = annotations1["label_discours"].replace('V-VERB_ACT', 'B-VERB_ACT')

textes_annotes = pd.unique(annotations1["doc_id_origine"])  # A modifier


In [4]:
# Création du df de travail pour le label_concret
annotations_work_concret1 = annotations1[["phrase_id", "token_text", "label_concret"]]
annotations_work_concret2 = annotations2[["phrase_id", "token_text", "label_concret"]]

# Clé identifiant le fichier d'origine
annotations_work_concret1['_src'] = 0
annotations_work_concret2['_src'] = 1

df_concat_concret = pd.concat([annotations_work_concret1, annotations_work_concret2], ignore_index=True)

# Renuméroter les 'phrase_id' en tenant compte de la source
df_concat_concret['phrase_id'] =(
    df_concat_concret.groupby(['_src', 'phrase_id'], sort=False)
    .ngroup() + 1
)

annotations_work_concret = df_concat_concret.groupby("phrase_id").agg(
    token_text=("token_text", list),
    label_concret=("label_concret", lambda x: [v[0] if isinstance(v, list) else v for v in x])
).reset_index()

In [30]:
# Création du df de travail pour le label_discours
annotations_work_disc1 = annotations1[["phrase_id", "token_text", "label_discours"]]
annotations_work_disc2 = annotations2[["phrase_id", "token_text", "label_discours"]]

# Clé identifiant le fichier d'origine
annotations_work_disc1['_src'] = 0
annotations_work_disc2['_src'] = 1

df_concat_disc = pd.concat([annotations_work_disc1, annotations_work_disc2], ignore_index=True)

# Renuméroter les 'phrase_id' en tenant compte de la source
df_concat_disc['phrase_id'] = (
    df_concat_disc.groupby(['_src', 'phrase_id'], sort=False)
    .ngroup() + 1
)
annotations_work_disc = df_concat_disc.groupby("phrase_id").agg(
    token_text=("token_text", list),
    label_discours=("label_discours", lambda x: [v[0] if isinstance(v, list) else v for v in x])
).reset_index()

In [6]:
# Préparation pour avoir une partie train et une partie test

def split_by_phrase_id(df, col_tag, train_ratio=0.8, val_ratio=0.1, seed=42):
    
    sentences = df.to_dict(orient="records")

    random.seed(seed)
    random.shuffle(sentences)

    n = len(sentences)

    def density_bucket(example):
        n_entities = sum(1 for t in example[col_tag] if t.startswith("B-"))
        if n_entities == 0:
            return 0
        elif n_entities <= 2:
            return 1
        else:
            return 2

    buckets = {0: [], 1: [], 2: []}
    for _, ex in df.iterrows():
        buckets[density_bucket(ex)].append(ex)

    train, val, test = [], [], []

    for bucket_data in buckets.values():
        random.shuffle(bucket_data)
        n = len(bucket_data)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)

        train += bucket_data[:n_train]
        val   += bucket_data[n_train:n_train + n_val]
        test  += bucket_data[n_train + n_val:]

    random.shuffle(train)
    random.shuffle(val)
    random.shuffle(test)

    print(f"Train : {len(train)} phrases")
    print(f"Val   : {len(val)} phrases")
    print(f"Test  : {len(test)} phrases")

    return train, val, test

## Travail sur le label concret

In [7]:
train, val, test = split_by_phrase_id(annotations_work_concret, "label_concret")

Train : 457 phrases
Val   : 56 phrases
Test  : 61 phrases


In [8]:
labels_conc = list({t for row in annotations_work_concret["label_concret"] for t in row})
num_labels_conc = len(labels_conc)
id2label = {id:label for id, label in enumerate(labels_conc)}
label2id = {label:id  for id, label in enumerate(labels_conc)}

def encode_labels(example, col_tag):
    example[col_tag] = [label2id[t] for t in example[col_tag]]
    return example

COL_TAG = "label_concret"
train = [row.to_dict() for row in train]
test  = [row.to_dict() for row in test]
val   = [row.to_dict() for row in val]

train_processed = [encode_labels(ex, COL_TAG) for ex in train]
test_processed = [encode_labels(ex, COL_TAG) for ex in test]
val_processed = [encode_labels(ex, COL_TAG) for ex in val]

train_dataset = Dataset.from_list(train_processed)
test_dataset = Dataset.from_list(test_processed)
val_dataset = Dataset.from_list(val_processed)

dataset = DatasetDict({"train": train_dataset, "test":test_dataset, "validation": val_dataset})

In [9]:
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification

MODEL_NAME = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["token_text"],
        col_tag = COL_TAG,
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )

    all_labels = []
    for i, labels in enumerate(examples[COL_TAG]):
        word_ids = tokenized.word_ids(batch_index=i)
        print(f"Nb labels: {len(labels)}, max word_id: {max(w for w in word_ids if w is not None)}")
        aligned_labels = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != prev_word_id:
                aligned_labels.append(labels[word_id])
            else:
                aligned_labels.append(-100)
            prev_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Map: 100%|██████████| 457/457 [00:00<00:00, 5663.06 examples/s]


Nb labels: 13, max word_id: 12
Nb labels: 25, max word_id: 24
Nb labels: 20, max word_id: 19
Nb labels: 76, max word_id: 75
Nb labels: 10, max word_id: 9
Nb labels: 45, max word_id: 44
Nb labels: 47, max word_id: 46
Nb labels: 54, max word_id: 53
Nb labels: 7, max word_id: 6
Nb labels: 16, max word_id: 15
Nb labels: 38, max word_id: 37
Nb labels: 29, max word_id: 28
Nb labels: 47, max word_id: 46
Nb labels: 62, max word_id: 61
Nb labels: 25, max word_id: 24
Nb labels: 56, max word_id: 55
Nb labels: 43, max word_id: 42
Nb labels: 79, max word_id: 78
Nb labels: 33, max word_id: 32
Nb labels: 66, max word_id: 65
Nb labels: 6, max word_id: 5
Nb labels: 27, max word_id: 26
Nb labels: 31, max word_id: 30
Nb labels: 49, max word_id: 48
Nb labels: 83, max word_id: 82
Nb labels: 1, max word_id: 0
Nb labels: 39, max word_id: 38
Nb labels: 68, max word_id: 67
Nb labels: 10, max word_id: 9
Nb labels: 26, max word_id: 25
Nb labels: 29, max word_id: 28
Nb labels: 51, max word_id: 50
Nb labels: 18, m

Map: 100%|██████████| 61/61 [00:00<00:00, 3307.25 examples/s]


Nb labels: 14, max word_id: 13
Nb labels: 21, max word_id: 20
Nb labels: 33, max word_id: 32
Nb labels: 32, max word_id: 31
Nb labels: 15, max word_id: 14
Nb labels: 50, max word_id: 49
Nb labels: 235, max word_id: 98
Nb labels: 39, max word_id: 38
Nb labels: 3, max word_id: 2
Nb labels: 58, max word_id: 57
Nb labels: 25, max word_id: 24
Nb labels: 18, max word_id: 17
Nb labels: 28, max word_id: 27
Nb labels: 43, max word_id: 42
Nb labels: 104, max word_id: 103
Nb labels: 45, max word_id: 44
Nb labels: 27, max word_id: 26
Nb labels: 38, max word_id: 37
Nb labels: 63, max word_id: 62
Nb labels: 35, max word_id: 34
Nb labels: 110, max word_id: 96
Nb labels: 21, max word_id: 20
Nb labels: 37, max word_id: 36
Nb labels: 26, max word_id: 25
Nb labels: 18, max word_id: 17
Nb labels: 71, max word_id: 70
Nb labels: 58, max word_id: 57
Nb labels: 71, max word_id: 70
Nb labels: 6, max word_id: 5
Nb labels: 61, max word_id: 60
Nb labels: 33, max word_id: 32
Nb labels: 29, max word_id: 28
Nb label

Map: 100%|██████████| 56/56 [00:00<00:00, 1565.84 examples/s]

Nb labels: 31, max word_id: 30
Nb labels: 110, max word_id: 96
Nb labels: 21, max word_id: 20
Nb labels: 39, max word_id: 38
Nb labels: 42, max word_id: 41
Nb labels: 32, max word_id: 31
Nb labels: 190, max word_id: 110
Nb labels: 183, max word_id: 102
Nb labels: 40, max word_id: 39
Nb labels: 45, max word_id: 44
Nb labels: 13, max word_id: 12
Nb labels: 63, max word_id: 62
Nb labels: 22, max word_id: 21
Nb labels: 18, max word_id: 17
Nb labels: 17, max word_id: 16
Nb labels: 13, max word_id: 12
Nb labels: 7, max word_id: 6
Nb labels: 38, max word_id: 37
Nb labels: 64, max word_id: 63
Nb labels: 15, max word_id: 14
Nb labels: 22, max word_id: 21
Nb labels: 78, max word_id: 77
Nb labels: 34, max word_id: 33
Nb labels: 80, max word_id: 79
Nb labels: 12, max word_id: 11
Nb labels: 67, max word_id: 66
Nb labels: 39, max word_id: 38
Nb labels: 8, max word_id: 7
Nb labels: 22, max word_id: 21
Nb labels: 49, max word_id: 48
Nb labels: 17, max word_id: 16
Nb labels: 8, max word_id: 7
Nb labels

In [20]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels_conc),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2816.72it/s]
CamembertForTokenClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
import torch
from collections import Counter
# Métrique seqeval (F1 par entité)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        p_row, l_row = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100: 
                p_row.append(id2label[p])
                l_row.append(id2label[l])
        true_preds.append(p_row)
        true_labels.append(l_row)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

all_labels_flat = [t for ex in train for t in ex["label_concret"]]
counts = Counter(all_labels_flat)
total  = sum(counts.values())
weights = torch.zeros(num_labels_conc)
for label_id, count in counts.items():
    weights[label_id] = total / (num_labels_conc * count)
print("Poids par classe :", {id2label[i]: round(w.item(), 2) for i, w in enumerate(weights)})
weights = torch.clamp(weights, max=10.0)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=weights.to(logits.device),
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, num_labels_conc), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Collator : gère le padding dynamique
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./camembert",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio = 0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=False,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Sauvegarde du meilleur modèle
trainer.save_model("./camembert-final")
tokenizer.save_pretrained("./camembert-final")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Poids par classe : {'I-VERB_CONCRET': 11.46, 'E-VERB_CONCRET': 31.71, 'B-VERB_CONTEXTE': 5.09, 'B-SUJ': 3.08, 'B-OBJ': 16.91, 'E-VERB_CONTEXTE': 53.8, 'I-VERB_CONTEXTE': 19.3, 'B-VERB_CONCRET': 6.8, 'O': 0.12}


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.255952,1.206683,0.342939,0.748428,0.470356,0.874885
2,0.766351,0.854430,0.383333,0.723270,0.501089,0.880385
3,0.532593,0.683064,0.508621,0.742138,0.603581,0.934005
4,0.357586,0.694120,0.624390,0.805031,0.703297,0.953254
5,0.465257,0.610311,0.595349,0.805031,0.684492,0.950504


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

('./camembert-final/tokenizer_config.json', './camembert-final/tokenizer.json')

In [22]:
# Courbes d'apprentissage du modèle
import json
import plotly.express as px 

with open("./camembert/checkpoint-290/trainer_state.json", "r") as file: 
    training_state = json.load(file)

loss = []

for log in training_state ["log_history"]:
    step = log["step"]
    if "loss" in log:
        loss += [{"step": step, "loss": log["loss"], "split": "train"}]
    elif "eval_loss" in log:
        loss += [{"step": step, "loss": log["eval_loss"], "split": "eval"}]
    else: 
        # thweird
        print(log)

loss = pd.DataFrame(loss)

px.line(loss, x = "step", y = "loss", color =  "split") 

In [23]:
labels_true = []
labels_pred = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

for batch in tokenized_dataset["test"].batch(batch_size=4, drop_last_batch=False):

    # Puis dans ta boucle d'inférence
    model_input = {
    "input_ids": torch.tensor(batch["input_ids"]).to(device),
    "attention_mask": torch.tensor(batch["attention_mask"]).to(device),}

    with torch.no_grad():
        logits = model(**model_input).logits

    predictions = torch.argmax(logits, dim=-1).cpu().numpy()
    gold_labels = np.array(batch["labels"])

    for pred_seq, label_seq in zip(predictions, gold_labels):
        for p, l in zip(pred_seq, label_seq):
            if l != -100:  # ✅ ignorer les tokens spéciaux et le padding
                labels_pred.append(id2label[int(p)])
                labels_true.append(id2label[int(l)])

(
    pd.DataFrame({
        "predict": labels_pred,
        "gold_standard": labels_true,
    })
    .to_csv("./outputs/predictions_ner.csv", index=False)
)

Batching examples: 100%|██████████| 61/61 [00:00<00:00, 1382.16 examples/s]


In [24]:
from sklearn.metrics import classification_report

print(classification_report(y_true = labels_true, y_pred = labels_pred))

                 precision    recall  f1-score   support

          B-OBJ       0.59      0.90      0.72        21
          B-SUJ       0.93      0.99      0.96        71
 B-VERB_CONCRET       0.60      0.86      0.70        29
B-VERB_CONTEXTE       0.82      0.82      0.82        45
 E-VERB_CONCRET       0.33      0.50      0.40         6
E-VERB_CONTEXTE       0.67      0.40      0.50        10
 I-VERB_CONCRET       0.33      0.44      0.38        18
I-VERB_CONTEXTE       0.55      0.38      0.44        16
              O       0.99      0.97      0.98      2023

       accuracy                           0.96      2239
      macro avg       0.65      0.70      0.66      2239
   weighted avg       0.96      0.96      0.96      2239



Meilleurs paramètres pour le moment :
training_args = TrainingArguments(
    output_dir="./camembert",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio = 0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=False,
)

## Travail sur le label_discours

In [32]:
train, val, test = split_by_phrase_id(annotations_work_disc, "label_discours")

Train : 458 phrases
Val   : 56 phrases
Test  : 60 phrases


In [33]:
labels_disc = list({t for row in annotations_work_disc["label_discours"] for t in row})
num_labels_disc = len(labels_disc)
id2label = {id:label for id, label in enumerate(labels_disc)}
label2id = {label:id  for id, label in enumerate(labels_disc)}

COL_TAG = "label_discours"
train = [row.to_dict() for row in train]
test  = [row.to_dict() for row in test]
val   = [row.to_dict() for row in val]

train_processed = [encode_labels(ex, COL_TAG) for ex in train]
test_processed = [encode_labels(ex, COL_TAG) for ex in test]
val_processed = [encode_labels(ex, COL_TAG) for ex in val]

train_dataset = Dataset.from_list(train_processed)
test_dataset = Dataset.from_list(test_processed)
val_dataset = Dataset.from_list(val_processed)

dataset = DatasetDict({"train": train_dataset, "test":test_dataset, "validation": val_dataset})

In [ ]:
MODEL_NAME = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["token_text"],
        col_tag = COL_TAG,
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )

    all_labels = []
    for i, labels in enumerate(examples[COL_TAG]):
        word_ids = tokenized.word_ids(batch_index=i)
        # print(f"Nb labels: {len(labels)}, max word_id: {max(w for w in word_ids if w is not None)}")
        aligned_labels = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != prev_word_id:
                aligned_labels.append(labels[word_id])
            else:
                aligned_labels.append(-100)
            prev_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

In [77]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels_disc),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4114.19it/s]
CamembertForTokenClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [78]:
seqeval = evaluate.load("seqeval")

all_labels_flat = [t for ex in train for t in ex["label_discours"]]
counts = Counter(all_labels_flat)
total  = sum(counts.values())
weights = torch.zeros(num_labels_disc)
for label_id, count in counts.items():
    weights[label_id] = total / (num_labels_disc * count)
print("Poids par classe :", {id2label[i]: round(w.item(), 2) for i, w in enumerate(weights)})
weights = torch.clamp(weights, max=10.0)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=weights.to(logits.device),
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, num_labels_disc), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Collator : gère le padding dynamique
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./camembert-DISC",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio = 0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=False,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Sauvegarde du meilleur modèle
trainer.save_model("./camembert-final-DISC")
tokenizer.save_pretrained("./camembert-final-DISC")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Poids par classe : {'I-VERB_ETAT': 27.67, 'E-VERB_ETAT': 108.83, 'E-VERB_ACT': 56.29, 'B_VERB-ETAT': 1632.4, 'B-SUJ': 2.87, 'B-OBJ': 14.98, 'I-VERB_ACT': 19.67, 'B-VERB_ETAT': 6.86, 'B-VERB_ACT': 4.52, 'O': 0.11}


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.063163,1.270731,0.441472,0.830189,0.576419,0.914664
2,0.723563,0.880657,0.386628,0.836478,0.528827,0.887200
3,0.456743,0.761391,0.555085,0.823899,0.663291,0.939186
4,0.322373,0.738626,0.741935,0.867925,0.800000,0.971064
5,0.565581,0.715606,0.724868,0.861635,0.787356,0.971064


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

('./camembert-final-DISC/tokenizer_config.json',
 './camembert-final-DISC/tokenizer.json')

In [79]:
with open("./camembert-DISC/checkpoint-290/trainer_state.json", "r") as file: 
    training_state = json.load(file)

loss = []

for log in training_state ["log_history"]:
    step = log["step"]
    if "loss" in log:
        loss += [{"step": step, "loss": log["loss"], "split": "train"}]
    elif "eval_loss" in log:
        loss += [{"step": step, "loss": log["eval_loss"], "split": "eval"}]
    else: 
        # thweird
        print(log)

loss = pd.DataFrame(loss)

px.line(loss, x = "step", y = "loss", color =  "split") 

In [80]:
labels_true = []
labels_pred = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

for batch in tokenized_dataset["test"].batch(batch_size=4, drop_last_batch=False):

    # Puis dans ta boucle d'inférence
    model_input = {
    "input_ids": torch.tensor(batch["input_ids"]).to(device),
    "attention_mask": torch.tensor(batch["attention_mask"]).to(device),}

    with torch.no_grad():
        logits = model(**model_input).logits

    predictions = torch.argmax(logits, dim=-1).cpu().numpy()
    gold_labels = np.array(batch["labels"])

    for pred_seq, label_seq in zip(predictions, gold_labels):
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                labels_pred.append(id2label[int(p)])
                labels_true.append(id2label[int(l)])

(
    pd.DataFrame({
        "predict": labels_pred,
        "gold_standard": labels_true,
    })
    .to_csv("./outputs/predictions_disc.csv", index=False)
)

print(classification_report(y_true = labels_true, y_pred = labels_pred))

Batching examples:   0%|          | 0/60 [00:00<?, ? examples/s]

Batching examples: 100%|██████████| 60/60 [00:00<00:00, 845.41 examples/s]


              precision    recall  f1-score   support

       B-OBJ       0.53      1.00      0.70        16
       B-SUJ       0.95      0.96      0.96        81
  B-VERB_ACT       0.80      0.86      0.83        57
 B-VERB_ETAT       0.66      0.75      0.70        28
  E-VERB_ACT       0.57      0.67      0.62         6
 E-VERB_ETAT       0.00      0.00      0.00         2
  I-VERB_ACT       0.53      0.50      0.52        16
 I-VERB_ETAT       0.42      0.83      0.56         6
           O       0.99      0.98      0.99      1938

    accuracy                           0.97      2150
   macro avg       0.61      0.73      0.65      2150
weighted avg       0.97      0.97      0.97      2150



/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Meilleurs paramètres :
training_args = TrainingArguments(
    output_dir="./camembert-DISC",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio = 0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=False,
)